In [ ]:
!pip install -q transformers
!pip install -q pytorch-lightning
!pip install -q wandb
!pip install -q torchmetrics

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from tqdm.auto import tqdm
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from transformers import T5Tokenizer, T5ForConditionalGeneration, get_cosine_schedule_with_warmup
from torch.optim import AdamW
from torchmetrics.text.rouge import ROUGEScore
from torchmetrics.text import BLEUScore
from pytorch_lightning import Trainer
import wandb
from sklearn.model_selection import train_test_split

In [ ]:
wandb.login(key='')
wandb.init(project="")

In [ ]:
class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        input_text = str(self.dataframe.iloc[index]['input']).strip()
        output_text = str(self.dataframe.iloc[index]['output']).strip()

        if not input_text or not output_text or not any(c in input_text for c in 'абвгдеёжзийклмнопрстуфхцчшщъыьэюяАБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ'):
            return self.__getitem__((index + 1) % len(self.dataframe))

        inputs = self.tokenizer.encode_plus(
            input_text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
            add_special_tokens=True
        )

        outputs = self.tokenizer.encode_plus(
            output_text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
            add_special_tokens=True
        )

        input_ids = inputs['input_ids'].squeeze()
        attention_mask = inputs['attention_mask'].squeeze()
        labels = outputs['input_ids'].squeeze()

        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'input_text': input_text,
            'output_text': output_text
        }

In [ ]:
class T5LightningModule(pl.LightningModule):
    def __init__(self, model_name='cointegrated/rut5-base-multitask', learning_rate=2e-5, total_steps=None):
        super().__init__()
        self.model = T5ForConditionalGeneration.from_pretrained(model_name)
        self.tokenizer = T5Tokenizer.from_pretrained(model_name, use_fast=True)
        self.learning_rate = learning_rate
        self.rouge_metric = ROUGEScore()
        self.bleu_metric = BLEUScore(n_gram=4, smooth=True)
        self.total_steps = total_steps
        self.save_hyperparameters()
        self.validation_step_outputs = []

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

    def training_step(self, batch, batch_idx):
        outputs = self(batch['input_ids'], batch['attention_mask'], batch['labels'])
        loss = outputs.loss

        if torch.isnan(loss) or torch.isinf(loss):
            return None

        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log('grad_norm', torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=0.5), on_step=True)
        return loss

    def validation_step(self, batch, batch_idx):
        outputs = self(batch['input_ids'], batch['attention_mask'], batch['labels'])
        loss = outputs.loss

        if torch.isnan(loss) or torch.isinf(loss):
            return {'val_loss': torch.tensor(0.0, device=self.device)}

        preds = self.model.generate(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

        preds_text = [self.tokenizer.decode(pred, skip_special_tokens=True, clean_up_tokenization_spaces=True) for pred in preds]
        labels_text = [batch['output_text'][i] for i in range(len(batch['output_text']))]
        input_text = [batch['input_text'][i] for i in range(len(batch['input_text']))]

        valid_pairs = [(p, l) for p, l in zip(preds_text, labels_text) if p.strip() and l.strip()]
        if valid_pairs:
            preds_text, labels_text = zip(*valid_pairs)
        else:
            preds_text, labels_text = [], []

        if batch_idx == 0:
            for i in range(min(len(preds_text), 3)):
                wandb.log({
                    f"val_example_{i}_input": input_text[i],
                    f"val_example_{i}_predicted": preds_text[i] if i < len(preds_text) else "<empty>",
                    f"val_example_{i}_reference": labels_text[i] if i < len(labels_text) else "<empty>",
                })

        self.validation_step_outputs.append({
            'val_loss': loss,
            'preds_text': preds_text,
            'labels_text': labels_text
        })

        return {'val_loss': loss}

    def on_validation_epoch_end(self):
        all_preds = []
        all_labels = []
        all_losses = []

        for output in self.validation_step_outputs:
            all_preds.extend(output['preds_text'])
            all_labels.extend(output['labels_text'])
            all_losses.append(output['val_loss'])

        avg_loss = torch.stack([l for l in all_losses if not torch.isnan(l)]).mean() if all_losses else torch.tensor(0.0, device=self.device)

        rouge1 = rouge2 = rougeL = bleu_score = 0.0
        if all_preds and all_labels:
            try:
                rouge_score = self.rouge_metric(list(all_preds), list(all_labels))
                rouge1 = rouge_score['rouge1_fmeasure'].mean().item()
                rouge2 = rouge_score['rouge2_fmeasure'].mean().item()
                rougeL = rouge_score['rougeL_fmeasure'].mean().item()
            except Exception as e:
                print(f"ROUGE computation error: {e}")

            try:
                bleu_score = self.bleu_metric(list(all_preds), [[label] for label in all_labels]).item()
            except Exception as e:
                print(f"BLEU computation error: {e}")

        self.log('val_loss', avg_loss, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log('rouge1', rouge1, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log('rouge2', rouge2, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log('rougeL', rougeL, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log('bleu', bleu_score, on_epoch=True, prog_bar=True, sync_dist=True)

        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=self.learning_rate, weight_decay=0.01, eps=1e-8)
        if self.total_steps:
            scheduler = get_cosine_schedule_with_warmup(
                optimizer,
                num_warmup_steps=int(0.1 * self.total_steps),
                num_training_steps=self.total_steps
            )
            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "interval": "step",
                }
            }
        return optimizer

def load_data(file_path_1, file_path_2):
    df = pd.read_csv(file_path_1)
    additional_df = pd.read_csv(file_path_2)

    df.rename(columns={'corrected': 'output', 'text': 'input'}, inplace=True)
    additional_df.rename(columns={'sent_correct': 'output', 'sent_corrupted': 'input'}, inplace=True)

    df = df[['input', 'output']].dropna()
    additional_df = additional_df[['input', 'output']].dropna()
    additional_df = additional_df.sample(frac=0.0, random_state=17)

    combined_df = pd.concat([df, additional_df], ignore_index=True)
    combined_df = combined_df[combined_df['input'].str.strip() != '']
    combined_df = combined_df[combined_df['output'].str.strip() != '']
    combined_df = combined_df[combined_df['input'].str.contains(r'[а-яА-Я]', na=False)]
    combined_df = combined_df[combined_df['output'].str.contains(r'[а-яА-Я]', na=False)]
    return combined_df

tokenizer = T5Tokenizer.from_pretrained("cointegrated/rut5-base-multitask", use_fast=True)

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("cointegrated/rut5-base-multitask", use_fast=True)

In [ ]:
def calc_token_len(example):
    tokens = tokenizer(example, return_tensors="pt", truncation=True, max_length=128)["input_ids"][0]
    return len(tokens)


In [ ]:
df = load_data('/kaggle/input/sentences/sentences.csv', '/kaggle/input/new-gen/new_gen.csv')
df["input"] = "Исправь грамматические ошибки в предложении: " + df["input"]
df["input_token_len"] = df["input"].apply(calc_token_len)
df = df[(df['input_token_len'] <= 61) & (df['input_token_len'] > 4)]

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.1, shuffle=True, random_state=42)
print(f"Train size: {len(train_df)}, Validation size: {len(val_df)}")

In [ ]:
train_dataset = TextDataset(train_df, tokenizer)
val_dataset = TextDataset(val_df, tokenizer)

batch_size = 16
num_epochs = 1
steps_per_epoch = len(train_dataset) // batch_size
total_steps = steps_per_epoch * num_epochs
print(f"Total training steps: {total_steps}")

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)


In [ ]:
model = T5LightningModule(
    model_name='cointegrated/rut5-base-multitask',
    learning_rate=3e-5,
    total_steps=total_steps
)

In [ ]:
trainer = Trainer(
    max_epochs=num_epochs,
    devices=1 if torch.cuda.is_available() else None,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    logger=pl.loggers.WandbLogger(project='t5_baseline_text_generation'),
    val_check_interval=0.5, 
    gradient_clip_val=0.5, 
    accumulate_grad_batches=1,
    precision=32,
    enable_checkpointing=True,
    callbacks=[
        pl.callbacks.ModelCheckpoint(
            monitor='val_loss',
            mode='min',
            save_top_k=1,  
            filename='t5-{epoch:02d}-{val_loss:.2f}'
        ),
        pl.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=3,
            mode='min',
            min_delta=0.001
        )
    ]
)

In [ ]:
trainer.fit(model, train_loader, val_loader)

In [ ]:
def generate_predictions(model, tokenizer, val_df, output_csv_path, batch_size=4, max_len=128, max_samples=100):
    val_df = val_df.sample(n=min(max_samples, len(val_df)), random_state=17)  
    val_dataset = TextDataset(val_df, tokenizer, max_len=max_len)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    inputs_text = []
    preds_text = []
    references_text = []

    model.eval()
    device = next(model.parameters()).device

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Generating predictions"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model.model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=max_len,
                num_beams=4,
                early_stopping=True,
                do_sample=False
            )

            preds = [tokenizer.decode(output, skip_special_tokens=True, clean_up_tokenization_spaces=True) for output in outputs]
            inputs = [batch['input_text'][i] for i in range(len(batch['input_text']))]
            references = [batch['output_text'][i] for i in range(len(batch['output_text']))]

            preds_text.extend(preds)
            inputs_text.extend(inputs)
            references_text.extend(references)

    results_df = pd.DataFrame({
        "input_text": inputs_text,
        "predicted_text": preds_text,
        "reference_text": references_text
    })
    results_df.to_csv(output_csv_path, index=False, encoding='utf-8')
    for i in range(min(5, len(results_df))):
        print(f"\nExample {i+1}:")
        print(f"Input: {results_df.iloc[i]['input_text']}")
        print(f"Reference: {results_df.iloc[i]['reference_text']}")
        print(f"Predicted: {results_df.iloc[i]['predicted_text']}")

    return results_df

In [ ]:
results_df = generate_predictions(model, tokenizer, val_df, 'output.csv')

In [ ]:
torch.save(model.model.state_dict(), 'model_t5.pt')
model.model.save_pretrained('t5_grammar_correction_model')
tokenizer.save_pretrained('t5_grammar_correction_model')

In [ ]:
try:
    rouge_metric = ROUGEScore()
    bleu_metric = BLEUScore(n_gram=4, smooth=True)

    valid_results = results_df[(results_df['predicted_text'].str.strip() != '') & (results_df['reference_text'].str.strip() != '')]
    if not valid_results.empty:
        rouge_scores = rouge_metric(valid_results['predicted_text'].tolist(), valid_results['reference_text'].tolist())
        bleu_score = bleu_metric(valid_results['predicted_text'].tolist(), [[ref] for ref in valid_results['reference_text'].tolist()])

        print(f"ROUGE-1: {rouge_scores['rouge1_fmeasure'].mean():.4f}")
        print(f"ROUGE-2: {rouge_scores['rouge2_fmeasure'].mean():.4f}")
        print(f"ROUGE-L: {rouge_scores['rougeL_fmeasure'].mean():.4f}")
        print(f"BLEU: {bleu_score:.4f}")
    else:
        print("No valid predictions for metrics computation.")
except Exception as e:
    print(f"Error computing final metrics: {e}")

wandb.finish()